# Comparison with freegs -- unsymmetrized (unstable solution)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BecerraMiguel/SemiFree-Solver/blob/main/notebooks/04_freegs_comparison_unsymmetrized.ipynb)


The semi-free solver is compared against `freegs`, the free-boundary solver underlying FreeGSNKE, using
**the same coil currents** found by the semi-free solver on each mesh and the same profiles $p(\psi_N)$ and
$g(\psi_N)$ as the fixed-boundary solver that produced $J_\phi$:
$p(\psi_N)=P_a\left[\left(1-(1-\psi_N)^2\right)^2+0.2\right]$, $g(\psi_N)=\psi_N^2$.

**No symmetry constraint is imposed here.** Because the plasma is strongly elongated ($\kappa=1.77$), the
symmetric equilibrium is prone to the vertical displacement instability: on four of the five meshes the
iteration converges, numerically stably, to a **vertically displaced** equilibrium ($Z<0$) that is
physically wrong and different from the semi-free one. The *asymmetry* and *axis Z* columns of the table
quantify it.

Notebook 05 repeats the study imposing the symmetry.


**Notebook flow.** (1) The $N_x=0$ semi-free solver is run on every mesh (results are reused if notebook 02
was already run on the same working directory). (2) `freegs` is run on every mesh; it requires `2**k+1`
points per axis, so it uses its own grid (65x129, 129x129, 129x257, 257x257, 257x513) and
$\psi_{semi-free}$ is interpolated onto it for the comparison. (3) The mean difference inside and outside
the plasma is computed, together with the run times.

**Estimated time (Colab, 2 cores):** semi-free ~45 min (if not already computed) + `freegs` 59, 295, 754, 1828 and 5399 s (2.3 h in total)
= **~3 hours**. Quick mode: ~25 min.


**DIII-D case** (same for every mesh): $R_0=1.67$ m, $a=0.67$ m, $\kappa=1.77$, $\delta=0.30$,
$I_p=1.5$ MA, $P_{axis}=50$ kPa, $B_{axis}=2.0$ T, $\Psi_b=0$, with $N_c=18$ PF coils.
Computational domain: $R\in[0.15, 3.0]$ m, $Z\in[-1.75, 1.75]$ m.

The inputs ($J_\phi$, boundary, coil positions) are shipped with the repository in `cases/DIII-D_<mesh>/`, and the
solver configuration in `configs/DIII-D_<mesh>.json`. This notebook clones the repository, builds the solver and
runs everything from there: nothing has to be uploaded.


**Options (first code cell):**
- `USE_GOOGLE_DRIVE = True` keeps the results in your Drive: if Colab disconnects, re-running the notebook reuses
the runs that already finished, and the notebooks share results with each other.
- `QUICK_MODE = True` uses only the 82x142, 100x172 and 151x261 meshes.

The values labelled *reference* are those obtained in earlier runs of the same study on Colab (2 cores). Timings
depend on the hardware; $\psi$ and the coil currents should agree with the reference up to rounding.



In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BecerraMiguel/SemiFree-Solver.git'
REPO_REF = None        # branch or tag to clone (None = default branch)
REPO_DIR = os.environ.get('REPRODUCE_REPO_DIR', '/content/SemiFree-Solver')
if not os.path.isdir(REPO_DIR):
    cmd = ['git', 'clone', '--depth', '1'] + (['--branch', REPO_REF] if REPO_REF else [])
    subprocess.run(cmd + [REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/notebooks')
import reproduce_common as rc
import numpy as np
import matplotlib.pyplot as plt

USE_GOOGLE_DRIVE = False   # True: keep results in Google Drive (survive disconnections, shared between notebooks)
QUICK_MODE = False       # True: only the three coarsest meshes (fast test; the order fits then use 2 points)
WORK = rc.default_work_dir(USE_GOOGLE_DRIVE)
os.makedirs(f'{WORK}/figures', exist_ok=True)
TAGS = rc.QUICK_TAGS if QUICK_MODE else rc.MESH_TAGS
print('Meshes:', TAGS)
rc.environment_report()
print('Working directory:', WORK)

## 1. Semi-free solver ($N_x=0$) on every mesh
Provides the coil currents and $\psi_{semi-free}$.

In [ ]:
BIN = rc.build_semifree(REPO_DIR, WORK, skip_bfield=True)
for tag in TAGS:
    rc.run_semifree(BIN, REPO_DIR, WORK, 'Nx0', tag)

## 2. Install `freegs`
A pinned commit is installed (it includes the NumPy 2.x compatibility fix missing from the PyPI release) and the profile translation is checked against finite differences.

In [ ]:
rc.install_freegs()
sys.path.insert(0, f'{REPO_DIR}/src/freegs_comparison')
rc.freegs_profile_selftest()

## 3. Run `freegs` on every mesh (without symmetrization)
Each mesh saves its result when it finishes; if the notebook is interrupted, re-running this cell continues where it stopped.

In [ ]:
OUT = 'freegs_unsymmetrized'
for tag in TAGS:
    rc.run_freegs_mesh(REPO_DIR, WORK, tag, symmetrize=False, out_subdir=OUT)

## 4. Comparison with the semi-free solver

In [ ]:
RES = {t: rc.load_freegs(REPO_DIR, WORK, t, OUT) for t in TAGS}
RES = {t: r for t, r in RES.items() if r is not None}
rc.freegs_table(RES, TAGS, 'unsym')

## 5. Fields per mesh: $\psi_{semi-free}$, $\psi_{freegs}$ and their difference
On the displaced meshes the core appears shifted towards $Z<0$ and the difference shows a vertical dipole.

In [ ]:
for tag, r in RES.items():
    rc.plot_freegs_crosscheck(r, fname=f'{WORK}/figures/freegs_unsym_{tag}.png',
                              label=' (unsymmetrized)')
    plt.show()

## 6. Mean difference versus mesh size

In [ ]:
rc.plot_freegs_error(REPO_DIR, RES, TAGS, fname=f'{WORK}/figures/freegs_unsym_error_vs_mesh.png')
plt.show()

## 7. Run times

In [ ]:
curves = {'freegs unsymmetrized (this run)': {t: r['elapsed_s'] for t, r in RES.items()},
          'freegs unsymmetrized (reference, Colab)': rc.REFERENCE['freegs_seconds']['unsym']}
rc.plot_freegs_timing(REPO_DIR, curves, fname=f'{WORK}/figures/freegs_unsym_timing.png')
plt.show()
for t, r in RES.items():
    print(f'{t:>10}: {r["elapsed_s"]:8.1f} s, {r["iterations"]} iterations, final relative change {r["relchange"]:.2e}')

## 8. Results bundle (optional)

In [ ]:
import glob
zp = rc.zip_results(WORK, ['freegs_unsymmetrized'], f'{WORK}/results_freegs_unsymmetrized.zip',
                    extra_files=sorted(glob.glob(f'{WORK}/figures/freegs_unsym_*.png')))
rc.offer_download(zp)